In [ ]:
from pathlib import Path
from trackmatexml import TrackmateXML
from matplotlib import pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import os
import re
from collections import defaultdict
import pickle
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

In [ ]:
# Read in .pkl file to create dictionary of tracks
output_dir = ''
with open(os.path.join(output_dir, 'Tracked_Bacteria_Dict.pkl'), 'rb') as f:
    ds_all_dict = pickle.load(f)

ds_all_dict



In [ ]:
#make sure all of the cells have tracks in the same format (ie arrays of cell dimensions)
for key, ds in ds_all_dict.items():
    track = ds.coords['track'] 
    if track.ndim ==0:
        ds = ds.assign_coords(track=("cell", np.repeat(track.item(), ds.sizes['cell'])))
    ds_all_dict[key]=ds

In [ ]:
def add_processed_data_to_dataset_dict(dataset_dict, data_var_to_modify, new_var_name, 
                                    interpolate = True, smooth = True, log_trans = False, 
                                     window_size = 3, save_dict=False):
    """
    goal here is to add a new data variable to the combined dataset dictionary
    dataset_dict is the dictionary of all the xarray datasets for each position
    data_var_to_modify is the data you want to you want to modify to create the new data variable
    new_var_name is name of the new data variable
    window_size is the size of moving window to use for the smoothing
    save_dict set to True if you want to save the updated dictionary - default is False - it is probably best to test things out first
    and you can always resave the dictionary outside of the function
    """

    for key, ds in dataset_dict.items():

        print(f"Running analysis on lineage: {key}")
        # ds is now our dataset - we don't need to select it
 
        # first get the GFP data or whatever variable you want to modify
        data = ds[data_var_to_modify]
        # normalize the data based on the background
        # interpolate the data - set fill_value to None to not go beyond edges I think
        if interpolate == True:
            data_interpolated = data.interpolate_na(dim='time', method='linear', fill_value=None,
                                                                    max_gap=1, use_coordinate=False)
        else:
            data_interpolated = data
        if smooth == True: 
            data_smoothed = data_interpolated.rolling(time=window_size, center=False, min_periods=1).mean()
        else:
            data_smoothed = data_interpolated
        
        # add the data to the ds
        ds[new_var_name] = data_smoothed
        #log transform and make a new array if you want one
        if log_trans ==True:
            ds[new_var_name+'_transformed'] = xr.apply_ufunc(np.log1p, ds[new_var_name])
        else:
            continue
    # save the data if save_dict = True
    if save_dict:
        with open(os.path.join(output_dir, 'all_datasets_dict.pkl'), 'wb') as f:
            pickle.dump(ds_all_dict, f)

    return dataset_dict


In [ ]:
#Applying GFP modifications for plot
add_processed_data_to_dataset_dict(ds_all_dict,  'GFP_median_intensity', 
                                    'GFP_median_intensity_processed', log_trans = False,  
                                    window_size = 3, save_dict=False)

In [ ]:
#plotting all cells, with selected cells highlighted as in figure
fig, ax = plt.subplots(figsize=(8, 5))
data_var='GFP_median_intensity_processed'
pos_to_color = ['20251105_XY017_final', '20251030_XY020_final', '20251030_XY016_final']
counter = 0
for key, ds in ds_all_dict.items():
    for track_name, ds_track in ds.groupby("track"):
        for cell in ds_track.cell.values:
            parent_value = ds_track.parents.sel(cell=cell)
            if parent_value ==0:
                var_values = ds_track[data_var].sel(cell=cell)
                non_nan_count = np.sum(~np.isnan(var_values))

                #print(non_nan_count)
                if non_nan_count > 12:
                  position = ds_track.position
                  if position.isin(pos_to_color) and track_name=='Track_0':
                    color = 'lime'
                    alpha = 1
                    linewidth = 1.5
                  else:
                     color = 'grey'
                     alpha = 0.5
                     linewidth = 1
                  counter += 1
                  style = 'solid' 
                  ax.plot(ds_track.time, var_values, color=color, alpha=alpha, linewidth = linewidth, linestyle= style)
                  ax.set_yscale('log')
                  ax.set_xlim(0, 240)
                  ax.set_xticks([0, 60, 120, 180, 240], minor=False)
                
            else:
                continue
print(counter)
plt.savefig(output_dir + '/Final_Tracks_Plot.pdf')

In [ ]:
#Plot individual tracks with their progeny (excluded in the plot above), rare cells have a division event that was trackable
data_var = 'GFP_median_intensity_processed'
# count how many tracks total
n_tracks = sum(
    len(ds.groupby("track"))
    for ds in ds_all_dict.values()
)

fig, axes = plt.subplots(
    n_tracks, 1,
    figsize=(4, 2 * n_tracks),
    sharex=True
)

# make axes iterable even if n_tracks == 1
axes = [axes] if n_tracks == 1 else axes

ax_idx = 0

for key, ds in ds_all_dict.items():
    for track_name, ds_track in ds.groupby("track"):

        ax = axes[ax_idx]

        for cell in ds_track.cell.values:
            var_values = ds_track[data_var].sel(cell=cell)
            parent_value = ds_track.parents.sel(cell=cell)
            if parent_value ==0:
                color = 'limegreen'
            else: 
                color = 'lightgrey' 
            alpha = 1 if ds_track.parents.sel(cell=cell)==0 else 0.5
            linewidth = 2 if ds_track.parents.sel(cell=cell)==0 else 0.75
            ax.plot(ds_track.time, var_values, color=color, alpha=alpha, linewidth = linewidth)
            ax.plot(ds_track.time, var_values, color=color, alpha=alpha, linewidth = linewidth)
            ax.set_xlim(0, 240)
            ax.set_ylim(100, 20000)
            ax.set_xticks([0, 60, 120, 180, 240], minor=False)
            

        ax.set_yscale("log")
        ax.set_title(f"{key} – Track {track_name}", fontsize=8)

        ax_idx += 1

plt.tight_layout()
plt.show()
